# Scaled-hydrograph runs — results

**Runs:** `..._warmstart_q050 / q075 / (baseline) / q125 / q150` — the July 2021 Ahr flood event
with all source hydrographs (and the matching spinup baseflow) multiplied by 0.5, 0.75, 1.0, 1.25, 1.5.
Each run is warm-started from its own scaled 6-day spinup (restart at 2021-07-12 00:00).

Created with `setup_scaled_runs.py` (July 2026), intended as training scenarios for the mSWE-GNN.



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

project_root = Path.cwd().parent  # repository root (run notebooks from notebooks/)
obs_dir      = project_root / 'data' / 'observations' / 'w'
mapping_file = project_root / 'data' / 'observations' / 'mapping.csv'
tref         = pd.Timestamp('2021-07-12 00:00:00')
hmin         = 0.05   # minimum depth to consider wet [m]
cell_area    = 100.0 * 100.0  # m2, for flooded-area stats

base = 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart'
runs = {
    0.50: project_root / 'models' / f'{base}_q050',
    0.75: project_root / 'models' / f'{base}_q075',
    1.00: project_root / 'models' / base,          # baseline
    1.25: project_root / 'models' / f'{base}_q125',
    1.50: project_root / 'models' / f'{base}_q150',
    2.00: project_root / 'models' / f'{base}_q200',
    3.00: project_root / 'models' / f'{base}_q300',
}
factors = sorted(runs)

# Scale factor is an ordered magnitude -> single-hue sequential ramp (light = 0.5x, dark = 1.5x).
# Observations keep black, so the darkest ramp step stays clearly blue.
ramp = plt.cm.Blues(np.linspace(0.35, 0.90, len(factors)))
run_colors = dict(zip(factors, ramp))
run_labels = {f: f'{f:.2f}x' + (' (baseline)' if f == 1.0 else '') for f in factors}

for f in factors:
    print(f'{run_labels[f]:18s} {runs[f].name:65s} exists={runs[f].exists()}')


def open_map(root):
    """Open sfincs_map.nc; decode_times=False because the timemax fill value
    breaks xarray's time decoding, then rebuild datetimes from tref."""
    ds = xr.open_dataset(root / 'sfincs_map.nc', decode_times=False, mask_and_scale=True)
    times = pd.to_datetime(ds['time'].values.astype(float), unit='s', origin=tref)
    return ds.assign_coords(time=times)


## 1. Water depth at t=0 (warm-start check)

Every scenario should show the river wet over its full length at the first timestep,
with slightly less water at lower factors (the spinup baseflow was scaled too).


In [ ]:
h0, peak_h = {}, {}
X = Y = None
for f in factors:
    ds = open_map(runs[f])
    h = (ds['zs'] - ds['zb']).where(ds['zb'] > -9999)
    h0[f] = h.isel(time=0).values
    peak_h[f] = h.max(dim='time', skipna=True).values   # reused in sections 3-5
    if X is None:
        X, Y = ds['x'].values, ds['y'].values
    ds.close()

# Crop all maps to the flood extent of the largest run (+ margin)
wet150 = peak_h[1.5] > hmin
xlim = (X[wet150].min() - 500, X[wet150].max() + 500)
ylim = (Y[wet150].min() - 500, Y[wet150].max() + 500)

print(f'{"factor":>7s} {"wet cells t=0":>14s} {"max depth t=0 [m]":>18s}')
for f in factors:
    print(f'{f:7.2f} {int(np.nansum(h0[f] > hmin)):14d} {np.nanmax(h0[f]):18.2f}')

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, f in zip(axes.flat, factors):
    plot = np.where(h0[f] > hmin, h0[f], np.nan)
    im = ax.pcolormesh(X, Y, plot, cmap='Blues', vmin=0, vmax=3, shading='nearest')
    ax.set_title(run_labels[f])
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('equal')
axes.flat[-1].axis('off')
fig.colorbar(im, ax=axes, label='Water depth [m]', shrink=0.7)
fig.suptitle(f'Water depth at t=0 ({tref:%Y-%m-%d %H:%M}) — shared color scale', fontsize=13)
plt.show()


## 2. Gauge station time series

Water depth at the observation points, all five scenarios on one axis per station
(single-hue ramp: light = 0.5x, dark = 1.5x). Observed depths (black) are plotted where
available — note they are only expected to match the 1.0x baseline.


In [ ]:
def load_his(root):
    ds = xr.open_dataset(root / 'sfincs_his.nc')
    raw = ds['station_name'].values
    names = []
    for row in raw:
        first = row[0]
        if isinstance(first, (bytes, np.bytes_)):
            names.append(b''.join(row).decode().strip())
        elif isinstance(first, (int, np.integer)):
            names.append(''.join(chr(c) for c in row if c != 0).strip())
        else:
            names.append(''.join(str(c) for c in row).strip())
    times = pd.DatetimeIndex(ds['time'].values)
    depth = ds['point_zs'].values - ds['point_zb'].values[np.newaxis, :]
    ds.close()
    return times, names, depth

his = {f: load_his(runs[f]) for f in factors}
station_names = his[1.0][1]
mapping = pd.read_csv(mapping_file, sep=';')

t_start = tref
t_end   = pd.Timestamp('2021-07-17 00:00')

for i, sname in enumerate(station_names):
    fig, ax = plt.subplots(figsize=(10, 4))
    for f in factors:
        times, _, depth = his[f]
        m = (times >= t_start) & (times <= t_end)
        ax.plot(times[m], depth[m, i], color=run_colors[f],
                linewidth=2.0 if f == 1.0 else 1.5, label=run_labels[f])

    row = mapping[mapping['name'] == sname]
    if not row.empty:
        obs_files = list(obs_dir.glob(f"{row['id'].iloc[0]}_*.csv"))
        if obs_files:
            df_obs = pd.read_csv(obs_files[0], sep=';', decimal=',',
                                 parse_dates=['Datum'], dayfirst=True, encoding='latin1')
            df_obs = df_obs[(df_obs['Datum'] >= t_start) & (df_obs['Datum'] <= t_end)]
            ax.plot(df_obs['Datum'], df_obs['Wert'] / 100.0,
                    label='Observed', color='black', linewidth=1.5)

    ax.set_title(f'Station {i}: {sname}')
    ax.set_ylabel('Water depth [m]'); ax.set_xlabel('Time')
    ax.grid(True, alpha=0.3); ax.legend(ncol=3, fontsize=9)
    plt.tight_layout(); plt.show()


## 3. Peak flood depth maps

Maximum water depth over the event, one panel per scenario on a **shared color scale**
so the panels are directly comparable.


In [ ]:
vmax_peak = float(np.ceil(np.nanpercentile(peak_h[1.5], 99.5)))
print(f'shared color scale: 0 to {vmax_peak:.0f} m '
      f'(99.5th percentile of the 1.50x run, absolute max {np.nanmax(peak_h[1.5]):.1f} m)')

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, f in zip(axes.flat, factors):
    plot = np.where(peak_h[f] > hmin, peak_h[f], np.nan)
    im = ax.pcolormesh(X, Y, plot, cmap='Blues', vmin=0, vmax=vmax_peak, shading='nearest')
    area_km2 = np.nansum(peak_h[f] > hmin) * cell_area / 1e6
    ax.set_title(f'{run_labels[f]} — {area_km2:.1f} km²')
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('equal')
axes.flat[-1].axis('off')
fig.colorbar(im, ax=axes, label='Peak water depth [m]', shrink=0.7)
fig.suptitle('Peak flood depth — shared color scale', fontsize=13)
plt.show()


## 4. Peak depth difference vs baseline

Where the scaling changes the flood: peak depth of each scenario minus the 1.0x baseline
(diverging scale, gray = no change). Cells wet in only one of the two runs are shown at
the difference from a dry bed (missing depth treated as 0).


In [ ]:
diff_factors = [f for f in factors if f != 1.0]
dmax = max(np.nanmax(np.abs(np.nan_to_num(peak_h[f]) - np.nan_to_num(peak_h[1.0])))
           for f in diff_factors)
dmax = float(np.ceil(dmax * 2) / 2)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, f in zip(axes.flat, diff_factors):
    diff = np.nan_to_num(peak_h[f]) - np.nan_to_num(peak_h[1.0])
    wet_either = (peak_h[f] > hmin) | (peak_h[1.0] > hmin)
    diff = np.where(wet_either, diff, np.nan)
    im = ax.pcolormesh(X, Y, diff, cmap='RdBu', vmin=-dmax, vmax=dmax, shading='nearest')
    ax.set_title(f'{run_labels[f]} minus baseline')
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('equal')
fig.colorbar(im, ax=axes, label='Peak depth difference [m] (blue = deeper than baseline)',
             shrink=0.7)
plt.show()


## 5. Summary table

One row per scenario. `peak depth @ most downstream station` uses the station with the
largest easting. Everything except the flood volume should increase strictly with the factor.


In [ ]:
ds_his = xr.open_dataset(runs[1.0] / 'sfincs_his.nc')
i_down = int(np.argmax(ds_his['station_x'].values))
ds_his.close()

rows = []
for f in factors:
    _, _, depth = his[f]
    rows.append({
        'factor': f,
        'wet cells t=0': int(np.nansum(h0[f] > hmin)),
        'flooded area [km2]': np.nansum(peak_h[f] > hmin) * cell_area / 1e6,
        'max peak depth [m]': float(np.nanmax(peak_h[f])),
        f'peak depth @ {station_names[i_down]} [m]': float(np.nanmax(depth[:, i_down])),
    })
summary = pd.DataFrame(rows).set_index('factor').round(2)

num = summary.select_dtypes('number')
mono = (num.diff().dropna() > 0).all()
print('strictly increasing with factor?')
print(mono.to_string(), '\n')
summary
